# 레슨 02 - Microsoft Agent Framework 탐색

<strong>Microsoft Agent Framework (MAF)</strong>는 AI 에이전트를 구축하기 위한 통합 프레임워크입니다. 네 가지 핵심 구성 요소로 깔끔하고 구성 가능한 아키텍처를 제공합니다:

- **Client** – AI 모델 엔드포인트에 연결하고 통신을 처리합니다
- **Agent** – 클라이언트를 래핑하여 지침과 도구 정의를 포함합니다
- **Tools** – 모델이 호출할 수 있는 사용자 정의 함수로 에이전트 기능을 확장합니다
- **Session** – 다회 대화를 위해 대화 기록을 유지합니다

이 레슨에서는 이러한 개념을 사용하여 목적지 가용성을 확인하는 <strong>여행 예약 에이전트</strong>를 만들어 봅니다.


## 설정


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## 에이전트 프레임워크 아키텍처 이해하기

Microsoft 에이전트 프레임워크는 계층화된 아키텍처를 따릅니다:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. <strong>클라이언트</strong> – `FoundryChatClient`는 Azure OpenAI 배포에 연결합니다. 인증 처리, 요청 형식화, 응답 파싱을 담당합니다.
2. <strong>에이전트</strong> – 클라이언트에서 `provider.create_agent()`를 통해 생성되며, 모델 접근과 지침(시스템 프롬프트), 도구를 결합합니다.
3. <strong>도구</strong> – `@tool`로 데코레이트된 Python 함수로, 에이전트가 호출하여 작업을 수행하거나 데이터를 검색할 수 있습니다.
4. <strong>세션</strong> – 대화 기록을 저장하는 `AgentSession` 객체로 (`agent.create_session()`을 통해 생성), 에이전트가 이전 문맥을 기억하여 다중 회차 대화를 가능하게 합니다.

각 계층을 단계별로 만들어 봅시다.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool 데코레이터로 도구 추가하기

도구는 에이전트가 텍스트 생성 이상의 작업을 수행하도록 합니다. `@tool` 데코레이터는 일반 파이썬 함수를 에이전트가 호출할 수 있는 것으로 변환합니다.

주요 사항:
- 모델이 각 매개변수를 이해할 수 있도록 `Annotated[type, "description"]`를 사용하세요.
- 도크스트링은 모델이 보는 도구 설명이 됩니다.
- `approval_mode="never_require"`는 사용자 확인 없이 도구가 자동으로 실행됨을 의미합니다.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 도구를 사용하여 에이전트 만들기

이제 클라이언트, 지침, 도구를 결합하여 에이전트를 만듭니다. `instructions`는 시스템 프롬프트 역할을 하며, 에이전트의 페르소나와 행동을 정의합니다.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 세션을 이용한 다중 대화

`AgentSession`( `agent.create_session()`을 통해 생성됨)는 대화 내 모든 메시지를 추적합니다. 동일한 세션을 각 `agent.run()` 호출에 전달함으로써, 에이전트는 전체 대화 기록에 접근할 수 있으며 이전 메시지를 참조할 수 있습니다.

각 회차마다 에이전트가 가용성 확인기를 호출할 수 있도록 `tools=[check_destination_availability]`를 전달합니다.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## 요약

이번 수업에서는 Microsoft 에이전트 프레임워크의 네 가지 기둥을 살펴보았습니다:

| 개념 | 학습 내용 |
|---------|------------------|
| <strong>클라이언트</strong> | `FoundryChatClient`가 자격 증명 기반 인증으로 Azure OpenAI에 연결합니다 |
| <strong>에이전트</strong> | `provider.create_agent()`가 모델 연결과 지침, 이름을 번들로 묶습니다 |
| <strong>도구</strong> | `@tool` 데코레이터가 에이전트가 호출할 Python 함수를 노출합니다 |
| <strong>세션</strong> | `agent.create_session()`이 여러 턴에 걸쳐 대화 기록을 유지합니다 |

이러한 구성 요소들이 결합되어 자연스러운 대화를 나누고, 외부 함수를 호출하며, 컨텍스트를 유지하는 에이전트를 만듭니다 — 이는 이후 수업에서 다룰 보다 고급 에이전트 패턴의 기반입니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
